In [ ]:

# Install BigQuery library
!pip install --upgrade google-cloud-bigquery

# Authenticate
from google.colab import auth
auth.authenticate_user()

# Import BigQuery client
from google.cloud import bigquery

# Setup BigQuery client
project_id = 'cool-benefit-456416-b0'
client = bigquery.Client(project=project_id)

# Define function to run query
def run_query(query):
    query_job = client.query(query)
    results = query_job.result()
    return results.to_dataframe()


### Van's Queries

In [ ]:
# Query 1: Most popular name by decade
query_van_1 = """
WITH Name_Decade AS (
  SELECT
    name,
    SUM(number) AS total_births,
    CAST(year/10 AS INT64)*10 AS decade  # Changed to CAST to INT64
  FROM `bigquery-public-data.usa_names.usa_1910_current`
  GROUP BY name, decade
)
SELECT decade, name, total_births
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY decade ORDER BY total_births DESC) as rn
  FROM Name_Decade
)
WHERE rn = 1
ORDER BY decade
"""
df_van_1 = run_query(query_van_1)
print("Van Query 1: Most Popular Name by Decade")
print(df_van_1)

Van Query 1: Most Popular Name by Decade
    decade     name  total_births
0     1910     Mary        161892
1     1920     Mary        680783
2     1930     Mary        638887
3     1940    James        650818
4     1950    James        867126
5     1960  Michael        872620
6     1970  Michael        784211
7     1980  Michael        684431
8     1990  Michael        600070
9     2000  Michael        331135
10    2010    Jacob        215500
11    2020     Liam        136044


In [ ]:

# Query 2: Gender-neutral names
query_van_2 = """
WITH Name_Gender AS (
  SELECT name, gender, SUM(number) AS total_births
  FROM `bigquery-public-data.usa_names.usa_1910_current`
  GROUP BY name, gender
)
SELECT name,
       SUM(CASE WHEN gender='M' THEN total_births ELSE 0 END) as male_count,
       SUM(CASE WHEN gender='F' THEN total_births ELSE 0 END) as female_count
FROM Name_Gender
GROUP BY name
HAVING ABS(SUM(CASE WHEN gender='M' THEN total_births ELSE 0 END) - SUM(CASE WHEN gender='F' THEN total_births ELSE 0 END)) < 1000
ORDER BY name
"""
df_van_2 = run_query(query_van_2)
print("Van Query 2: Gender Neutral Names")
print(df_van_2)


Van Query 2: Gender Neutral Names
           name  male_count  female_count
0         Aaban          12             0
1         Aadam           6             0
2         Aadan          23             0
3       Aadarsh          11             0
4        Aadhav          43             0
...         ...         ...           ...
26366     Zyria           0            81
26367    Zyriah           0            63
26368     Zyron           5             0
26369     Zyrus           5             0
26370  Zyshonne           5             0

[26371 rows x 3 columns]


### Stephen's Queries

In [ ]:

# Query 1: States with most unique names
query_stephen_1 = """
SELECT state, COUNT(DISTINCT name) AS unique_names
FROM `bigquery-public-data.usa_names.usa_1910_current`
GROUP BY state
ORDER BY unique_names DESC
LIMIT 10
"""
df_stephen_1 = run_query(query_stephen_1)
print("Stephen Query 1: States with Most Unique Names")
print(df_stephen_1)


Stephen Query 1: States with Most Unique Names
  state  unique_names
0    CA         20239
1    TX         18238
2    NY         14997
3    FL         11869
4    IL         11493
5    GA         10284
6    OH          9844
7    PA          9582
8    NC          9345
9    MI          9178


In [ ]:

# Query 2: Fastest rising names after 1990
query_stephen_2 = """
WITH Name_90s AS (
  SELECT name, SUM(number) AS total_90s
  FROM `bigquery-public-data.usa_names.usa_1910_current`
  WHERE year BETWEEN 1990 AND 1999
  GROUP BY name
),
Name_2000s AS (
  SELECT name, SUM(number) AS total_2000s
  FROM `bigquery-public-data.usa_names.usa_1910_current`
  WHERE year BETWEEN 2000 AND 2009
  GROUP BY name
)
SELECT n2.name, n1.total_90s, n2.total_2000s, (n2.total_2000s - n1.total_90s) AS increase
FROM Name_90s n1
JOIN Name_2000s n2 ON n1.name = n2.name
ORDER BY increase DESC
LIMIT 10
"""
df_stephen_2 = run_query(query_stephen_2)
print("Stephen Query 2: Fastest Rising Names after 1990")
print(df_stephen_2)


Stephen Query 2: Fastest Rising Names after 1990
       name  total_90s  total_2000s  increase
0     Ethan      66914       201931    135017
1  Isabella      18484       149591    131107
2      Emma      58213       181425    123212
3       Ava       3691       104534    100843
4   Madison      93465       193665    100200
5    Jayden       3506       102417     98911
6    Sophia      24296       119246     94950
7      Noah      56232       143677     87445
8     Aiden       1268        84046     82778
9    Olivia      75902       156104     80202


### Andy's Queries

In [ ]:

# Query 1: Celebrity influence - 'Kobe'
query_andy_1 = """
SELECT year, SUM(number) AS total_births
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE name = 'Kobe'
GROUP BY year
ORDER BY year
"""
df_andy_1 = run_query(query_andy_1)
print("Andy Query 1: Celebrity Influence - 'Kobe' Name Trend")
print(df_andy_1)


Andy Query 1: Celebrity Influence - 'Kobe' Name Trend
    year  total_births
0   1996            37
1   1997           304
2   1998          1093
3   1999           829
4   2000          1425
5   2001          1538
6   2002          1383
7   2003          1200
8   2004           594
9   2005           412
10  2006           429
11  2007           535
12  2008           706
13  2009           650
14  2010           622
15  2011           510
16  2012           512
17  2013           519
18  2014           484
19  2015           422
20  2016           550
21  2017           509
22  2018           439
23  2019           473
24  2020          1493
25  2021          1101


In [ ]:

# Query 2: Stable names across decades
query_andy_2 = """
WITH Name_Year AS (
  SELECT name, year, SUM(number) AS total_births
  FROM `bigquery-public-data.usa_names.usa_1910_current`
  GROUP BY name, year
),
Name_Decade_Avg AS (
  SELECT name, (year/10)*10 AS decade, AVG(total_births) as avg_births
  FROM Name_Year
  GROUP BY name, decade
)
SELECT name, COUNT(decade) AS decades_appeared
FROM Name_Decade_Avg
WHERE avg_births > 1000
GROUP BY name
ORDER BY decades_appeared DESC
LIMIT 10
"""
df_andy_2 = run_query(query_andy_2)
print("Andy Query 2: Most Stable Names Across Decades")
print(df_andy_2)


Andy Query 2: Most Stable Names Across Decades
      name  decades_appeared
0    Henry               112
1   Edward               112
2     John               112
3   George               112
4     Paul               112
5     Jack               112
6   Joseph               112
7  William               112
8   Thomas               112
9   Samuel               112
